<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Spiral_of_Theodorus_Geometric_Animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Spiral of Theodorus: Geometric Animation

## Overview
This notebook contains a Python implementation for generating a high-quality animation of the Spiral of Theodorus (also known as the Pythagorean Spiral). The animation illustrates the step-by-step construction of contiguous right triangles, providing both a visual and mathematical representation of irrational square roots.

## Mathematical Principles
The Spiral of Theodorus is constructed using the following geometric rules:
1. The construction begins with an isosceles right triangle with legs of length 1.
2. The hypotenuse of the first triangle (length sqrt(2)) serves as the base leg for the second triangle.
3. The outer leg of every subsequent triangle maintains a constant length of 1.
4. This process repeats, with the nth triangle having a base leg of length sqrt(n) and a hypotenuse of length sqrt(n+1).

Historically, Theodorus of Cyrene used this spiral to prove the irrationality of the square roots of non-square integers from 3 up to 17.

## Technical Implementation
The script utilizes the following stack:
- **Matplotlib**: Used for the vector-based rendering of geometric shapes and mathematical labels using LaTeX-style formatting.
- **OpenCV (cv2)**: Used to compile individual rendered frames into a high-definition MP4 video file.
- **NumPy**: Handles the trigonometric calculations required to determine the vertex coordinates for each triangle.
- **IPython Display**: Enables the direct embedding and playback of the resulting MP4 video within the Colab environment.

## Credits
- **Author**: Mugambi Ndwiga
- **Project**: Crafts and Engineering
- **Instagram**: @craftsandengineering

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import cv2
import os
import shutil
from IPython.display import HTML
from base64 import b64encode

"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Premise: The Spiral of Theodorus Animation

Mathematical and Historical Context:
The Spiral of Theodorus, also known as the Pythagorean Spiral, is a construction of contiguous
right triangles. It begins with an isosceles right triangle with legs of length 1.
The hypotenuse of each triangle becomes the base leg of the next triangle, while the outer
leg always maintains a constant length of 1.

Key Properties:
- Hypotenuse Lengths: The lengths of the hypotenuses follow the sequence sqrt(1), sqrt(2), ..., sqrt(n).
- Historical Significance: Named after Theodorus of Cyrene (5th Century BC), who used this
  construction to prove the irrationality of the square roots of non-square integers from 3 to 17.
- Growth: As n increases, the spiral winds outwards, with the total accumulated angle
  growing proportionally to the square root of n.
"""

def create_spiral_animation(output_file='spiral_theodorus_v3.mp4', num_steps=15):
    frames_dir = 'frames_v3'
    # Clear existing frames to ensure the video length is dynamic to ONLY this run
    if os.path.exists(frames_dir): shutil.rmtree(frames_dir)
    os.makedirs(frames_dir)

    fps = 10
    linger_frames = fps * 1

    fig, ax = plt.subplots(figsize=(10, 10), facecolor='black')
    points = [(1, 0)]
    current_angle = 0

    frame_count = 0
    for i in range(1, num_steps + 1):
        p_prev = points[-1]
        dist = np.sqrt(p_prev[0]**2 + p_prev[1]**2)
        angle_step = np.arctan(1/dist)
        current_angle += angle_step
        p_new = (np.sqrt(dist**2 + 1) * np.cos(current_angle), np.sqrt(dist**2 + 1) * np.sin(current_angle))
        points.append(p_new)

        for phase in ['triangle', 'label']:
            for f in range(linger_frames):
                ax.clear()
                ax.set_facecolor('black')
                limit = 5
                ax.set_xlim(-limit, limit)
                ax.set_ylim(-limit, limit)
                ax.axis('off')

                for j in range(1, len(points)):
                    poly = Polygon([(0,0), points[j-1], points[j]], closed=True,
                                   facecolor=plt.cm.plasma(j/num_steps), edgecolor='white', alpha=0.6, linewidth=0.5)
                    ax.add_patch(poly)

                for j in range(1, i + (1 if phase == 'label' else 0)):
                    pp = points[j-1]
                    pn = points[j]
                    ax.text(pp[0]*0.6, pp[1]*0.6, f'√{j}', color='white', fontsize=10, ha='center', fontweight='bold')
                    ax.text((pp[0]+pn[0])/2 * 1.15, (pp[1]+pn[1])/2 * 1.15, '1', color='cyan', fontsize=10, ha='center', fontweight='bold')

                ax.text(0.5, 0.95, 'Spiral of Theodorus', color='gold', fontsize=22, ha='center', transform=ax.transAxes, weight='bold')
                ax.text(0.98, 0.02, '@craftsandengineering', color='white', alpha=0.3, fontsize=8, transform=ax.transAxes, ha='right')

                plt.savefig(f'{frames_dir}/f_{frame_count:05d}.png', dpi=80, facecolor='black')
                frame_count += 1

    # Linger on final scene for 2 seconds
    for _ in range(fps * 2):
        plt.savefig(f'{frames_dir}/f_{frame_count:05d}.png', dpi=80, facecolor='black')
        frame_count += 1

    # Show closing title card for 2 seconds
    for _ in range(fps * 2):
        ax.clear()
        ax.set_facecolor('black')
        ax.axis('off')
        ax.text(0.5, 0.6, 'Mathematical Beauty', color='gold', fontsize=28, ha='center', transform=ax.transAxes, weight='bold')
        ax.text(0.5, 0.45, 'The Spiral of Theodorus', color='white', fontsize=18, ha='center', transform=ax.transAxes)
        ax.text(0.5, 0.2, '@craftsandengineering', color='cyan', fontsize=12, ha='center', transform=ax.transAxes)
        plt.savefig(f'{frames_dir}/f_{frame_count:05d}.png', dpi=80, facecolor='black')
        frame_count += 1

    plt.close()
    images = sorted([img for img in os.listdir(frames_dir) if img.endswith(".png")])
    frame = cv2.imread(os.path.join(frames_dir, images[0]))
    h, w, _ = frame.shape
    video = cv2.VideoWriter(output_file, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for img_name in images:
        video.write(cv2.imread(os.path.join(frames_dir, img_name)))
    video.release()

    with open(output_file, "rb") as f: video_base64 = b64encode(f.read()).decode()
    return display(HTML(f'<div style="text-align:center"><video width="600" controls><source src="data:video/mp4;base64,{video_base64}" type="video/mp4"></video><br><a href="data:video/mp4;base64,{video_base64}" download="{output_file}" style="color:#00AAFF; font-weight:bold; text-decoration:none;">Download MP4</a></div>'))

create_spiral_animation()